In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
df = pd.read_json("../../../data/post_quality/raw/post_data_v4.json")
df.reset_index(drop=True)
df.head()

,id,title,body,effort,openness,is_confident,subreddit,tag
0,1,If your first-ever attempt at gambling went co...,,0,0,True,r/Showerthoughts,NaN
1,2,"According to birds, all land animals are botto...",,0,0,True,r/Showerthoughts,NaN
2,3,Very few people can actually keep their watch ...,,0,0,True,r/Showerthoughts,NaN
3,4,"What is keeping the really deadly diseases, li...",,0,0,True,r/askscience,NaN
4,5,When did humans start living indoors?,,0,0,True,r/askscience,NaN


Dropping some unwanted column

In [3]:
df.drop(columns=['effort','is_confident','subreddit','tag','id'],axis=1, inplace=True)

In [4]:
df.tail()

,title,body,openness
235,Does being an introvert actually lead to highe...,There’s a common belief that introverts tend t...,1
236,Does anything seem legendary anymore?,I was having a conversation with a friend as t...,1
237,How will US government deal with a large propo...,How will US government deal with a large propo...,1
238,Why so many straight men are constantly polici...,I’ve noticed a pattern where straight guys lov...,1
239,Are we done?,Imagine the year is 2050 AI has evolved into A...,1


In [5]:
for col in df.columns:
    print(f"Number of null value in '{col}' column is {df[col].isnull().sum()}")

Number of null value in 'title' column is 0
Number of null value in 'body' column is 0
Number of null value in 'openness' column is 0


In [6]:
import numpy as np

def cosine_sim(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

In [7]:
from sklearn.model_selection import StratifiedKFold
from sklearn.linear_model import LogisticRegression
import pandas as pd
import numpy as np

def run_k_fold_embedding_error_analysis(
    df: pd.DataFrame,
    embedding_col: str,
    label_col: str = "openness",
    n_splits: int = 5,
    model_name: str = "embedding_semantic",
    k_neighbors: int = 3
):
    """
    Runs k-fold CV for a frozen-embedding + linear model,
    with semantic error analysis.
    """

    skf = StratifiedKFold(
        n_splits=n_splits,
        shuffle=True,
        random_state=42
    )

    all_rows = []

    for fold, (train_idx, test_idx) in enumerate(skf.split(df, df[label_col])):
        train_df = df.iloc[train_idx]
        test_df = df.iloc[test_idx]

        X_train = np.stack(train_df[embedding_col].values)
        y_train = train_df[label_col].values

        X_test = np.stack(test_df[embedding_col].values)
        y_test = test_df[label_col].values

        model = LogisticRegression(
            max_iter=1000,
            class_weight="balanced",
            random_state=42
        )
        model.fit(X_train, y_train)

        probs = model.predict_proba(X_test)[:, 1]
        preds = (probs >= 0.5).astype(int)

        # ----- class centroids (interpretability) -----
        centroid_open = X_train[y_train == 1].mean(axis=0)
        centroid_closed = X_train[y_train == 0].mean(axis=0)

        # ----- nearest neighbors (semantic explainability) -----
        for i, row in test_df.iterrows():
            emb = row[embedding_col]

            sims = np.array([
                cosine_sim(emb, e) for e in X_train
            ])

            nn_idx = sims.argsort()[-k_neighbors:][::-1]
            nn_labels = y_train[nn_idx]

            all_rows.append({
                "fold": fold,
                "true_label": row[label_col],
                "pred_label": preds[test_df.index.get_loc(i)],
                "prob_openness": probs[test_df.index.get_loc(i)],
                "correct": preds[test_df.index.get_loc(i)] == row[label_col],

                # centroid diagnostics
                "sim_to_open_centroid": cosine_sim(emb, centroid_open),
                "sim_to_closed_centroid": cosine_sim(emb, centroid_closed),

                # nearest-neighbor diagnostics
                "nn_label_distribution": nn_labels.tolist(),
                "nn_open_ratio": nn_labels.mean(),

                # raw text (for manual inspection)
                "title": row.get("title", ""),
                "body": row.get("body", "")
            })

    results_df = pd.DataFrame(all_rows)

    errors_only = results_df[results_df["correct"] == False]

    return {
        "results_all": results_df,
        "errors_only": errors_only,
        "error_group_counts": (
            errors_only
            .groupby(["true_label", "pred_label"])
            .size()
        ),
        "model_name": model_name
    }

In [8]:
def make_combined_text(row):
    title = row["title"].strip() if pd.notna(row["title"]) else ""
    body = row["body"].strip() if pd.notna(row["body"]) else ""

    if title and body:
        return f"Title: {title}\n\nBody: {body}"
    elif title:
        return f"Title: {title}"
    else:
        return body

In [9]:
df['combined_text'] = df.apply(make_combined_text,axis=1)

In [10]:
df.head()

,title,body,openness,combined_text
0,If your first-ever attempt at gambling went co...,,0,Title: If your first-ever attempt at gambling ...
1,"According to birds, all land animals are botto...",,0,"Title: According to birds, all land animals ar..."
2,Very few people can actually keep their watch ...,,0,Title: Very few people can actually keep their...
3,"What is keeping the really deadly diseases, li...",,0,Title: What is keeping the really deadly disea...
4,When did humans start living indoors?,,0,Title: When did humans start living indoors?


In [11]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")


c:\Users\prakh\projects\discusso-ml\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [12]:
texts = df["combined_text"].tolist()

embeddings = model.encode(
    texts,
    batch_size=32,
    show_progress_bar=True,
    normalize_embeddings=True
)
df["embedding"] = list(embeddings)

Batches: 100%|██████████| 8/8 [00:03<00:00,  2.44it/s]


In [13]:
# shape check
df["embedding"].iloc[0].shape
# → (384,)


(384,)

In [14]:

# no NaNs
np.isnan(df["embedding"].iloc[0]).any()
# → False

np.False_

In [15]:
df[["combined_text", "embedding"]].head(2)

,combined_text,embedding
0,Title: If your first-ever attempt at gambling ...,"[0.01007612, 0.057246886, -0.013716099, -0.017..."
1,"Title: According to birds, all land animals ar...","[0.07378493, 0.0049494402, 0.07531181, -0.0445..."


In [16]:
results = run_k_fold_embedding_error_analysis(
    df=df,
    embedding_col="embedding",
    label_col="openness"
)

In [17]:
results

{'results_all':      fold  true_label  pred_label  prob_openness  correct  \
 0       0           0           1       0.681100    False   
 1       0           0           0       0.327262     True   
 2       0           0           1       0.579457    False   
 3       0           0           0       0.199537     True   
 4       0           0           0       0.339680     True   
 ..    ...         ...         ...            ...      ...   
 235     4           1           0       0.496307    False   
 236     4           1           1       0.672195     True   
 237     4           1           1       0.658179     True   
 238     4           1           1       0.780806     True   
 239     4           1           1       0.802090     True   
 
      sim_to_open_centroid  sim_to_closed_centroid nn_label_distribution  \
 0                0.229552                0.086424             [1, 1, 1]   
 1                0.157728                0.281928             [0, 0, 1]   
 2         

In [18]:
errors_v1 = results["errors_only"]

In [45]:
all_res_v1 = results["results_all"]
true_label_v1 = all_res_v1[all_res_v1["correct"] == True]

In [47]:
true_label_v1.shape

(204, 11)

In [19]:
errors_v1.shape

(36, 11)

In [20]:
sample_v1 = errors_v1.sample(25,random_state=42)

In [21]:
sample_v1.reset_index()

,index,fold,true_label,pred_label,prob_openness,correct,sim_to_open_centroid,sim_to_closed_centroid,nn_label_distribution,nn_open_ratio,title,body
0,235,4,1,0,0.496307,False,0.176695,0.150542,"[1, 1, 1]",1.000000,CMV: Emergency sirens and car horns should not...,"First, some definitions\nBy “emergency sirens,..."
1,116,2,0,1,0.525379,False,0.150153,0.087432,"[1, 1, 1]",1.000000,Is it possible for branching ratio and deviati...,"Hey everyone, I’m a beginner in computational ..."
2,170,3,1,0,0.474155,False,0.071679,0.050760,"[1, 1, 1]",1.000000,Where does the last name “Whelchel” come from,The person is from European immigrants definit...
3,209,4,0,1,0.705263,False,0.193165,0.005392,"[1, 1, 1]",1.000000,"If Yellowstone erupted, what would the lasting...",Considering factors like aerosols that would r...
4,127,2,1,0,0.347132,False,0.233156,0.351270,"[0, 0, 0]",0.000000,DAE ever get a intense overwhelming feeling of...,
5,212,4,0,1,0.620929,False,0.083201,-0.039017,"[1, 1, 1]",1.000000,What is an Arab? When African and Asian countr...,"Hi,\n\nAs a Saudi, I don't feel like I have mu..."
6,147,3,0,1,0.508294,False,0.062707,0.011137,"[0, 0, 1]",0.333333,There aren't any girl cereal mascots.,
7,113,2,0,1,0.500315,False,0.002112,-0.025132,"[0, 1, 1]",0.666667,Can I travel to US on (B1/B2) with expired Ger...,"Hello all,\nI would like to know if I can trav..."
8,76,1,1,0,0.441529,False,0.187041,0.266569,"[1, 0, 1]",0.666667,"$3 million, but you can never have penetrating...",You are not physically able to have sexual int...
9,128,2,1,0,0.460683,False,0.182673,0.247506,"[1, 1, 0]",0.666667,DAE else ever fall asleep with a video on loop...,


## Error taxonomy

In [22]:
sample_v1["error_bucket"] = ""
sample_v1["notes"] = ""

In [23]:
# Borderline/Weakly Framed Open Posts
# Hypothetical question misinterpreted as closed
# Factual / Binary-Answer Questions Misclassified as Open
# Label Ambiguity
# Personal experience/story/venting
# Statement misinterpreted as closed
sample_v1.loc[sample_v1.index[0], "error_bucket"] = "Borderline/Weakly Framed Open Posts"
sample_v1.loc[sample_v1.index[0], "notes"] = (
    "This post actually has CMV tag, but there is no other explicit sign for openness. "
    "Model got confused whether this post is open and asking for opinion or just shower thought types."
)

sample_v1.loc[sample_v1.index[1], "error_bucket"] = "Factual / Binary-Answer Questions Misclassified as Open"
sample_v1.loc[sample_v1.index[1], "notes"] = "This post is specifically of a domain, and asking a factual question. Model misinterpreted the interogative nature as openness."

sample_v1.loc[sample_v1.index[2], "error_bucket"] = "Borderline/Weakly Framed Open Posts"
sample_v1.loc[sample_v1.index[2], "notes"] = "This post is asking about the origin of a last name. Many people can give many different answers. But Model got confused between whether it is a factual post starting with where.... or open opinion based"

sample_v1.loc[sample_v1.index[3], "error_bucket"] = "Factual / Binary-Answer Questions Misclassified as Open"
sample_v1.loc[sample_v1.index[3], "notes"] = "This post is asking about a geological question. The response will be converging on one/two factual answers. Model misinterpreted the interogative tone."

sample_v1.loc[sample_v1.index[4], "error_bucket"] = "Label Ambiguity"
sample_v1.loc[sample_v1.index[4], "notes"] = "This post can be debatable whether it is asking about just yes/no answer, or explicit asking for experiences. I personally think it may go with label 0."

sample_v1.loc[sample_v1.index[5], "error_bucket"] = "Factual / Binary-Answer Questions Misclassified as Open"
sample_v1.loc[sample_v1.index[5], "notes"] = "This post is talking about a communities definition, Same as 3rd index row."

sample_v1.loc[sample_v1.index[6], "error_bucket"] = "Borderline/Weakly Framed Open Posts"
sample_v1.loc[sample_v1.index[6], "notes"] = "This post is just a sentence. Model got confused when it is asking for opinion on this or just a simple statement."

sample_v1.loc[sample_v1.index[7], "error_bucket"] = "Factual / Binary-Answer Questions Misclassified as Open"
sample_v1.loc[sample_v1.index[7], "notes"] = "This is just asking for a suggestion. There can only be 1 or 2 responses. Model misunderstood it as open post."

sample_v1.loc[sample_v1.index[8], "error_bucket"] = "Borderline/Weakly Framed Open Posts"
sample_v1.loc[sample_v1.index[8], "notes"] = "This is a hypothetical scenario asking post. It is loosely framed open ended post. Model got confused here."

sample_v1.loc[sample_v1.index[9], "error_bucket"] = "Borderline/Weakly Framed Open Posts"
sample_v1.loc[sample_v1.index[9], "notes"] = "Model got confused it as closed post, due to it's loosely made open-ended."

sample_v1.loc[sample_v1.index[10], "error_bucket"] = "Borderline/Weakly Framed Open Posts"
sample_v1.loc[sample_v1.index[10], "notes"] = "Model confused it as a closed, even though this post can have many suggestions to recover from religious trauma."

sample_v1.loc[sample_v1.index[11], "error_bucket"] = "Hypothetical question misinterpreted as closed"
sample_v1.loc[sample_v1.index[11], "notes"] = "This post is asking about a hypothetical question by taking a historical context. Model misinterpreted it as a closed post"

sample_v1.loc[sample_v1.index[12], "error_bucket"] = "Statement misinterpreted as closed"
sample_v1.loc[sample_v1.index[12], "notes"] = "Writer just wrote a sentence, nothing about views, advice, opinion. Model thought it as a open"

sample_v1.loc[sample_v1.index[13], "error_bucket"] = "Label Ambiguity"
sample_v1.loc[sample_v1.index[13], "notes"] = "Binary label can be debatable here."

sample_v1.loc[sample_v1.index[14], "error_bucket"] = "Factual / Binary-Answer Questions Misclassified as Open"
sample_v1.loc[sample_v1.index[14], "notes"] = "Looks open, it is a venting, but actually closed."


sample_v1.loc[sample_v1.index[15], "error_bucket"] = "Borderline/Weakly Framed Open Posts"
sample_v1.loc[sample_v1.index[15], "notes"] = "Model confused on this post, as it is weakly framed open post. Post must also include somthing like 'What's your opinion...'"


sample_v1.loc[sample_v1.index[16], "error_bucket"] = "Borderline/Weakly Framed Open Posts"
sample_v1.loc[sample_v1.index[16], "notes"] = "This post can generate many different answers. Model confused thinking it is closed."

sample_v1.loc[sample_v1.index[17], "error_bucket"] = "Borderline/Weakly Framed Open Posts"
sample_v1.loc[sample_v1.index[17], "notes"] = "Same as above one"

sample_v1.loc[sample_v1.index[18], "error_bucket"] = "Factual / Binary-Answer Questions Misclassified as Open"
sample_v1.loc[sample_v1.index[18], "notes"] = "Similar logic as 3rd row(0 indexing)"

sample_v1.loc[sample_v1.index[19], "error_bucket"] = "Personal experience/story"
sample_v1.loc[sample_v1.index[19], "notes"] = "Suprisingly I think it must be the noise, because by using semantic models, ranting, venting, personal stories, are reduced significantly."

sample_v1.loc[sample_v1.index[20], "error_bucket"] = "Factual / Binary-Answer Questions Misclassified as Open"
sample_v1.loc[sample_v1.index[20], "notes"] = "Same as the 4th row(0 indexing)"

sample_v1.loc[sample_v1.index[21], "error_bucket"] = "Personal experience/story/venting"
sample_v1.loc[sample_v1.index[21], "notes"] = "Same as 19th row(0 indexing)"


sample_v1.loc[sample_v1.index[22], "error_bucket"] = "Borderline/Weakly Framed Open Posts"
sample_v1.loc[sample_v1.index[22], "notes"] = "Hypothetical question confused model and by small margin it predicted wrong answer"

sample_v1.loc[sample_v1.index[23], "error_bucket"] = "Factual / Binary-Answer Questions Misclassified as Open"
sample_v1.loc[sample_v1.index[23], "notes"] = "Same logic as 3rd row of this table(0 indexing)"

sample_v1.loc[sample_v1.index[24], "error_bucket"] = "Borderline/Weakly Framed Open Posts"
sample_v1.loc[sample_v1.index[24], "notes"] = "Loose framing of DAE question made model thought it as closed."


In [24]:
sample_v1.iloc[24]

fold                                                                      4
true_label                                                                1
pred_label                                                                0
prob_openness                                                      0.489107
correct                                                               False
sim_to_open_centroid                                               0.207236
sim_to_closed_centroid                                             0.227378
nn_label_distribution                                             [1, 0, 0]
nn_open_ratio                                                      0.333333
title                     DAE open an app without meaning to and suddenl...
body                      Sometimes I’ll check my phone for one thing an...
error_bucket                            Borderline/Weakly Framed Open Posts
notes                     Loose framing of DAE question made model thoug...
Name: 222, d

In [25]:
sample_v1["error_bucket"].value_counts()

error_bucket
Borderline/Weakly Framed Open Posts                        11
Factual / Binary-Answer Questions Misclassified as Open     8
Label Ambiguity                                             2
Hypothetical question misinterpreted as closed              1
Statement misinterpreted as closed                          1
Personal experience/story                                   1
Personal experience/story/venting                           1
Name: count, dtype: int64

Here is your content converted into **clean Markdown format** for a Jupyter Notebook (with proper headings, bullet points, and tables where useful):

---

# Phase 1: Semantic Baseline – Error Analysis Summary

## Overview

After training a **frozen sentence-embedding + logistic regression classifier** using **5-fold cross-validation**, I conducted a manual review of **25 misclassified samples** to understand the structure of model errors.

* The semantic model produced significantly fewer total errors (**36 across all folds**) compared to previous TF-IDF and surface-level baselines.
* Most incorrect predictions had probability scores near the decision boundary (**0.4–0.6**), indicating calibrated uncertainty.

Below is a breakdown of observed error categories.

---

# Error Analysis Buckets

## Error Bucket 1: Borderline / Weakly-Framed Open Posts (~30–35%)

### Description

Posts that are syntactically open-ended but pragmatically shallow:

* “Would you rather…”
* “DAE…”
* Hypothetical prompts without deeper perspective-seeking intent
* Closed questions framed in open form

### Observation

* Model confidence typically between **0.45–0.55**
* Errors occur near the decision boundary

### Interpretation

These errors reflect inherent ambiguity in the *openness construct*.

The model’s uncertainty aligns with semantic overlap between open and closed discourse.

These are acceptable errors due to construct fuzziness.

---

## Error Bucket 2: Factual / Binary-Answer Questions Misclassified as Open (~35–40%)

### Description

* Domain-specific factual questions
* Questions likely to receive yes/no or short factual responses
* Limited invitation for personal belief, experience, or interpretation

### Observation

* Often predicted as **open (label = 1)** with relatively high confidence
* Model appears to overvalue interrogative structure

### Interpretation

The semantic model captures **“question-ness”** but does not fully distinguish between:

* Interrogative form
* Perspective-seeking intent

This represents a **systematic bias** rather than random error and requires further investigation.

---

## Error Bucket 3: Label Ambiguity (~7–10%)

### Description

* Posts that could reasonably belong to either label
* Boundary cases between open and closed discourse

### Interpretation

This reflects inherent subjectivity in the labeling process and suggests a **non-zero level of unavoidable disagreement**.

---

# Confidence-Based Analysis

* Approximately **30 out of 36 total errors** fall within a probability range of **0.4–0.6**
* Abstaining in this confidence interval would significantly reduce incorrect predictions

## Conclusion

The semantic model appears **well-calibrated**:

* Uncertainty aligns with ambiguous cases
* Errors are not arbitrary failures

---

# Error Distribution Summary

| Error Bucket | Description                                      | Estimated Share | Key Pattern                                   |
| ------------ | ------------------------------------------------ | --------------- | --------------------------------------------- |
| Bucket 1     | Borderline / weakly-framed open posts            | ~30–35%         | Near decision boundary (0.45–0.55 confidence) |
| Bucket 2     | Factual / binary questions misclassified as open | ~35–40%         | High-confidence false positives               |
| Bucket 3     | Label ambiguity                                  | ~7–10%          | Inherent subjectivity                         |

---

# Phase 1 Conclusion

The semantic baseline demonstrates that:

* Openness has a meaningful semantic signal in embedding space
* Most errors cluster near decision boundaries
* Systematic misclassification occurs primarily in factual/binary interrogatives
* Confidence scores correlate with ambiguity

---

## Next Step

Investigate **latent pragmatic intent dimensions** that distinguish:

* Factual questions
* Perspective-seeking openness



# Intent Probing



# Define 3 Pragmatic Intent Axes (Formally)

We want axes that explain your error buckets.

Keep them **simple**, **interpretable**, and **defensible**.

---

## Axis 1: Factual-Seeking vs Perspective-Seeking

### **Formal Definition**

* **Factual-Seeking:**
  The post primarily requests **verifiable information** about the world.

* **Perspective-Seeking:**
  The post primarily invites **subjective beliefs, experiences, interpretations, or opinions**.

### **Why this axis?**

Bucket 2 errors appear *surface-factual* but were predicted as open-ended.
This axis tests whether the model is confusing:

* Information requests
  vs
* Opinion invitations

---

## Axis 2: Binary-Answer vs Elaborative-Answer

### **Formal Definition**

* **Binary-Answer Framing:**
  The expected or typical response can be a **yes/no answer or short factual statement**.

* **Elaborative-Answer Framing:**
  The expected response requires **explanation, reasoning, justification, or narrative detail**.

### **Why this axis?**

Many errors appear to expect **short, constrained answers**,
but the model predicts them as discussion-style prompts.

This axis captures **response-length expectation mismatch**.

---

## Axis 3: Hypothetical / Counterfactual Framing

### **Formal Definition**

* **Hypothetical Framing:**
  The post presents imagined, speculative, or counterfactual scenarios.

* **Non-Hypothetical Framing:**
  The post refers to real-world facts, events, or lived experiences.

### **Why this axis?**

Posts like *“Would you rather…”* clustered in Bucket 1.
This suggests boundary ambiguity between:

* Open-ended discussion
  vs
* Structured speculative prompts

This axis may explain classification instability.

---

# Create Simple Heuristic Probes

We are **NOT** training a new model.

We are building **rough indicators**.

They do **not** need to be perfect.

They only need to **correlate meaningfully** with your error buckets.

### Guiding Principle

Heuristics should be:

* Interpretable
* Lightweight
* Reproducible
* Easy to justify in analysis

The goal is **explanatory power**, not predictive perfection.



In [26]:
df["text_lower"] = df["combined_text"].str.lower()

#### **Axis 1 Heuristic: Perspective-Seeking Indicator**

In [27]:
perspective_keywords = [
    "why do you",
    "what do you think",
    "how do you feel",
    "in your opinion",
    "what's your experience",
    "do you believe",
    "from your perspective"
]

df["perspective_flag"] = df["text_lower"].apply(
    lambda x: any(k in x for k in perspective_keywords)
)

#### **Axis 2 Heuristic: Binary-Answer Indicator**

In [28]:
binary_starters = [
    "should i",
    "can i",
    "am i",
    "is it",
    "do you",
    "does anyone",
    "would you"
]

def binary_question_flag(text):
    return any(text.strip().startswith(starter) for starter in binary_starters)

df["binary_flag"] = df["text_lower"].apply(binary_question_flag)
df["yes_no_phrase"] = df["text_lower"].str.contains("yes or no")
df["binary_indicator"] = df["binary_flag"] | df["yes_no_phrase"]

#### **Axis 3 Heuristic: Hypothetical Indicator**

In [29]:
hypothetical_keywords = [
    "would you rather",
    "what if",
    "imagine",
    "suppose",
    "hypothetically"
]

df["hypothetical_flag"] = df["text_lower"].apply(
    lambda x: any(k in x for k in hypothetical_keywords)
)

### Now let's test our hypothesis from the above observation:
* Many high confidence FP are factual binary answer question.
* The model over estimated interogative structure

In [59]:
high_conf_fp = errors_v1[(errors_v1["true_label"] == 0) & (errors_v1['pred_label'] == 1)]
high_conf_tp = true_label_v1[(true_label_v1["true_label"] == 1) & (true_label_v1['pred_label'] == 1)]
high_conf_fn = errors_v1[(errors_v1["true_label"] == 1) & (errors_v1['pred_label'] == 0)]
high_conf_tn = true_label_v1[(true_label_v1["true_label"] == 0) & (true_label_v1['pred_label'] == 0)]

In [60]:
high_conf_fp['combined_text'] = high_conf_fp.apply(make_combined_text,axis=1)
high_conf_tp['combined_text'] = high_conf_tp.apply(make_combined_text,axis=1)
high_conf_fn['combined_text'] = high_conf_fn.apply(make_combined_text,axis=1)
high_conf_tn['combined_text'] = high_conf_tn.apply(make_combined_text,axis=1)

C:\Users\prakh\AppData\Local\Temp\ipykernel_25056\1712025243.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  high_conf_fp['combined_text'] = high_conf_fp.apply(make_combined_text,axis=1)
C:\Users\prakh\AppData\Local\Temp\ipykernel_25056\1712025243.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  high_conf_tp['combined_text'] = high_conf_tp.apply(make_combined_text,axis=1)
C:\Users\prakh\AppData\Local\Temp\ipykernel_25056\1712025243.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy 

In [61]:
analysis_df = high_conf_fp.merge(
    df[["combined_text", "binary_indicator", "perspective_flag", "hypothetical_flag"]],
    on="combined_text",
    how="left"
)

analysis_df_tp = high_conf_tp.merge(
    df[["combined_text", "binary_indicator", "perspective_flag", "hypothetical_flag"]],
    on="combined_text",
    how="left"
)

analysis_df_fn = high_conf_fn.merge(
    df[["combined_text", "binary_indicator", "perspective_flag", "hypothetical_flag"]],
    on="combined_text",
    how="left"
)

analysis_df_tn = high_conf_tn.merge(
    df[["combined_text", "binary_indicator", "perspective_flag", "hypothetical_flag"]],
    on="combined_text",
    how="left"
)

In [52]:
high_conf_tp.shape

(104, 12)

In [53]:
analysis_df_tp.shape

(104, 15)

In [62]:
high_conf_tn.shape,analysis_df_tn.shape

((100, 12), (100, 15))

In [63]:
high_conf_fn.shape,analysis_df_fn.shape

((16, 12), (16, 15))

In [64]:
high_conf_fp.shape, analysis_df.shape

((20, 12), (20, 15))

In [33]:
analysis_df["binary_indicator"].mean()


np.float64(0.0)

In [51]:
analysis_df_tp["binary_indicator"].mean()


np.float64(0.0)

In [34]:
analysis_df["perspective_flag"].mean()

np.float64(0.0)

In [35]:

analysis_df["hypothetical_flag"].mean()

np.float64(0.0)

In [36]:
analysis_df[["binary_indicator", 
             "perspective_flag", 
             "hypothetical_flag"]].isna().mean()


binary_indicator     0.0
perspective_flag     0.0
hypothetical_flag    0.0
dtype: float64

In [37]:
analysis_df[["combined_text"]].head()

,combined_text
0,Title: Very few people can actually keep their...
1,"Title: Just realized I've been saying ""epitome..."
2,Title: The snyderverse subreddit is like a cul...
3,Title: Are people born with different artery s...
4,"Title: According to birds, all land animals ar..."


In [38]:
analysis_df["combined_text"].nunique()
high_conf_fp["combined_text"].nunique()

20

In [39]:
df["text_lower"].head()

0    title: if your first-ever attempt at gambling ...
1    title: according to birds, all land animals ar...
2    title: very few people can actually keep their...
3    title: what is keeping the really deadly disea...
4         title: when did humans start living indoors?
Name: text_lower, dtype: object

In [40]:
analysis_df.shape

(20, 15)

In [41]:
high_conf_fp.shape

(20, 12)

In [42]:
analysis_df

,fold,true_label,pred_label,prob_openness,correct,sim_to_open_centroid,sim_to_closed_centroid,nn_label_distribution,nn_open_ratio,title,body,combined_text,binary_indicator,perspective_flag,hypothetical_flag
0,0,0,1,0.681100,False,0.229552,0.086424,"[1, 1, 1]",1.000000,Very few people can actually keep their watch ...,,Title: Very few people can actually keep their...,False,False,False
1,0,0,1,0.579457,False,0.252006,0.218334,"[1, 1, 0]",0.666667,"Just realized I've been saying ""epitome"" wrong...","Been saying ""epi-TOME"" like it rhymes with hom...","Title: Just realized I've been saying ""epitome...",False,False,False
2,0,0,1,0.759979,False,0.322910,0.101159,"[1, 1, 1]",1.000000,The snyderverse subreddit is like a cult.,Some of the rules saying you can’t disrespect ...,Title: The snyderverse subreddit is like a cul...,False,False,False
3,0,0,1,0.641805,False,0.095855,-0.095084,"[1, 1, 1]",1.000000,Are people born with different artery size?,I’m wondering if some people are just genetica...,Title: Are people born with different artery s...,False,False,False
4,1,0,1,0.514835,False,0.081530,0.004483,"[0, 1, 1]",0.666667,"According to birds, all land animals are botto...",,"Title: According to birds, all land animals ar...",False,False,False
5,1,0,1,0.520021,False,0.181819,0.133130,"[0, 1, 1]",0.666667,When did humans start living indoors?,,Title: When did humans start living indoors?,False,False,False
6,2,0,1,0.537407,False,0.109981,0.025139,"[0, 0, 0]",0.000000,"What is keeping the really deadly diseases, li...",,Title: What is keeping the really deadly disea...,False,False,False
7,2,0,1,0.500315,False,0.002112,-0.025132,"[0, 1, 1]",0.666667,Can I travel to US on (B1/B2) with expired Ger...,"Hello all,\nI would like to know if I can trav...",Title: Can I travel to US on (B1/B2) with expi...,False,False,False
8,2,0,1,0.525379,False,0.150153,0.087432,"[1, 1, 1]",1.000000,Is it possible for branching ratio and deviati...,"Hey everyone, I’m a beginner in computational ...",Title: Is it possible for branching ratio and ...,False,False,False
9,2,0,1,0.679514,False,0.147354,-0.005462,"[1, 1, 1]",1.000000,Did 14th century knights wear eyepatches?,Im reading Thomas Asbridge’s biography of Will...,Title: Did 14th century knights wear eyepatche...,False,False,False


In [72]:
## For tp
analysis_df_tp['binary_indicator'].value_counts(),analysis_df_tp['hypothetical_flag'].value_counts(),analysis_df_tp['perspective_flag'].value_counts()

(binary_indicator
 False    104
 Name: count, dtype: int64,
 hypothetical_flag
 False    88
 True     16
 Name: count, dtype: int64,
 perspective_flag
 False    100
 True       4
 Name: count, dtype: int64)

In [71]:
## For fp
analysis_df['binary_indicator'].value_counts(),analysis_df['hypothetical_flag'].value_counts(),analysis_df['perspective_flag'].value_counts()

(binary_indicator
 False    20
 Name: count, dtype: int64,
 hypothetical_flag
 False    20
 Name: count, dtype: int64,
 perspective_flag
 False    20
 Name: count, dtype: int64)

In [73]:
## For fn
analysis_df_fn['binary_indicator'].value_counts(),analysis_df_fn['hypothetical_flag'].value_counts(),analysis_df_fn['perspective_flag'].value_counts()

(binary_indicator
 False    16
 Name: count, dtype: int64,
 hypothetical_flag
 False    13
 True      3
 Name: count, dtype: int64,
 perspective_flag
 False    16
 Name: count, dtype: int64)

In [74]:
## For tn
analysis_df_tn['binary_indicator'].value_counts(),analysis_df_tn['hypothetical_flag'].value_counts(),analysis_df_tn['perspective_flag'].value_counts()

(binary_indicator
 False    100
 Name: count, dtype: int64,
 hypothetical_flag
 False    96
 True      4
 Name: count, dtype: int64,
 perspective_flag
 False    100
 Name: count, dtype: int64)

# Lexical Pragmatic Intent Probing

## Pragmatic Axes Defined

We analyzed three lexical-pragmatic dimensions:

* **Binary framing markers**
* **Perspective invitation markers**
* **Hypothetical framing markers**

Their distribution was computed across the following groups:

| Group | Description     |
| ----- | --------------- |
| TP    | True Positives  |
| FP    | False Positives |
| FN    | False Negatives |
| TN    | True Negatives  |

---

## Observations

### 1. Binary Framing Markers

* Absent across **all groups**.

### 2. Perspective Invitation Markers

* Extremely sparse (<5%).
* No observable correlation with error types.

### 3. Hypothetical Framing Markers

* More common in **True Positives (~15%)**.
* Less common in **True Negatives (~4%)**.
* Suggests an association with openness or exploratory framing.
* **Not overrepresented in False Positives**.

---

## Interpretation

* Embedding misclassification does **not** appear to be driven by explicit lexical pragmatic framing.
* Surface-level markers do not meaningfully explain model error patterns.
* Errors likely stem from:

  * Deeper structural ambiguity
  * Semantic complexity
  * Discourse-level dynamics

---

## Conclusion

Lexical-level intent probing is **insufficient** to explain false positive behavior.

Further investigation should focus on:

* Structural analysis
* Discourse-level modeling
* Higher-order semantic ambiguity

---


# Final Conclusion – Openness Model

## *Summary*

The final openness model uses **frozen text embeddings** combined with a **linear classifier**.  

Stratified *k-fold evaluation* shows stable performance across folds, with relatively few **high-confidence misclassifications**.

---

## *Error Analysis*

Error inspection revealed:

- Most misclassifications occur near the decision boundary (*probability ≈ 0.5*), indicating semantic ambiguity rather than systematic bias.  
- High-confidence false positives are rare.  
- Lexical pragmatic probing (*binary framing, perspective invitation, hypothetical framing*) did not reveal significant overrepresentation in false positive cases.

---

## *Interpretation*

The embedding-based model captures semantic openness signals effectively within dataset constraints.  

Misclassifications appear to stem from inherent ambiguity in the openness definition rather than clear structural or lexical deficiencies.

---

## *Calibration Insight*

The model demonstrates useful confidence behavior:

- Incorrect predictions are often associated with lower confidence.  

This supports the possibility of implementing an *abstention threshold* for safer deployment.

---

## *Decision*

Given:

- Low high-confidence error rate  
- No clear systematic failure pattern  
- Diminishing returns from further refinement  

We conclude that the openness model is sufficiently mature for integration into the project pipeline.

Further modeling effort will focus on the **Effort model**, with openness treated as a stable upstream component.
